In [1]:
import contextlib
from pathlib import Path
from uuid import uuid4

import plotly.figure_factory as ff
import plotly.graph_objects as go
import polars as pl
import torch
import torch.nn.functional as F
from IPython.display import display
from lightning import Fabric
from plotly.subplots import make_subplots
from tqdm.autonotebook import tqdm

from src.chart_transport.constraint import (
    LagrangianConstraintConfig,
    LossConstraintConfig,
)
from src.config.base import BaseConfig
from src.data.gaussian_mixture.data import MultimodalGaussianDataConfig
from src.experiments.multimodal_gaussian.canonical import (
    get_canonical_chart_transport_configs,
)
from src.experiments.multimodal_gaussian.config import MultimodalGaussianTrainingConfig
from src.experiments.multimodal_gaussian.plotting.constraints import (
    plot_latents,
    plot_sample_pairs,
)

torch.set_float32_matmul_precision("medium")

In [2]:
num_modes = 8
mode_std = 0.1
offset = 0.0
latent_dimension = 2
ambient_dimension = 128

data_config = MultimodalGaussianDataConfig.initialize(
    num_modes=num_modes,
    mode_std=mode_std,
    offset=offset,
    ambient_dimension=ambient_dimension,
)

chart_transport_config = get_canonical_chart_transport_configs(
    data_config=data_config,
    latent_dimension=latent_dimension
)

training_config = MultimodalGaussianTrainingConfig.initialize(
    seed=0,
    train_batch_size=4096,
    eval_batch_size=4096,
    integrated_n_steps=1_000_000,
    folder=(
        Path("/home/nlyu/Code/diffusive-latent-generation/artifacts/multimodal_gaussian")
        / uuid4().hex
    ),
)

chart_transport_config.visualize()

In [3]:
fabric = Fabric(
    accelerator="cuda", devices=1,
    precision="bf16-mixed",
)
fabric.launch()
fabric.seed_everything(training_config.seed)

device = fabric.device
device_type = device.type

runtime_data_config = data_config.replace(
    path="isometry",
    replacement=data_config.isometry.to(device=device, dtype=torch.float32),
).replace(
    path="projection",
    replacement=data_config.projection.to(device=device, dtype=torch.float32),
)

architecture_config = chart_transport_config.architecture_config
chart_transport_model = architecture_config.get_model()
optimizer = architecture_config.get_optimizer(
    model=chart_transport_model,
)
chart_transport_model, optimizer = fabric.setup(chart_transport_model, optimizer)

encoder = chart_transport_model.encoder
decoder = chart_transport_model.decoder
critic = chart_transport_model.critic

prior_config = chart_transport_config.prior_config
constraint_config = chart_transport_config.loss_config.constraint_config
constraint_method = constraint_config.constraint_method
pretrain_config = chart_transport_config.loss_config.chart_pretrain_config
transport_config = chart_transport_config.loss_config.transport_config
transport_t_min, transport_t_max = transport_config.t_range

data_dual = torch.zeros((), device=device)
prior_dual = torch.zeros((), device=device)

fabric.print(
    f"device={device}, precision=bf16-mixed, folder={training_config.folder}"
)

Using bfloat16 Automatic Mixed Precision (AMP)
Seed set to 0


device=cuda:0, precision=bf16-mixed, folder=/home/nlyu/Code/diffusive-latent-generation/artifacts/multimodal_gaussian/90b92f677bda4edc9eaa4f342d6751e2


In [4]:
class ConstraintMonitorConfig(BaseConfig):
    plot_n_sample_pairs: int
    plot_n_data_latents_per_mode: int


class CriticMonitorConfig(BaseConfig):
    """
    Config for visualizing the score implied by the critic. By one plot, we mean "html + png"
    1. Saves one plot for each sample_t_values, with two arrows attached to each point
        corresponding to the "data score" and the "prior - data score" at the noise level
    2. Saves one plot "transport "
        corresponding to the implied drift field for **clean data latent**,
        averaged across the noise spectrum. Background corresponds to contour lines
    3. Saves one plot "loss_spectrum" with x-axis being losses at
        unif[0, 1, interval=0.05] corresponding to the **score matching** loss at each point.
        Read the theory very carefully, I think this corresponds to noise-pred mse * (1-t)**2 / t ** 2
    Keep these specs after modification
    """

    sample_t_values: list[float]
    num_contour_lines: int
    plot_n_data_latents_per_mode: int
    plot_n_vectors_per_mode: int


class IntegratedMonitorConfig(BaseConfig):
    """
    Integrated training needs to execute constraint-monitor, critic-monitor, together with:
    save a scatterplot of the generated samples.
    Also save plotly plots of:
    1. Dual variables (data + prior) on the right axis, together with loss (data / prior) annotated on the left axis
    2. Transport field norm at step (right axis), avg log likelihood of batch samples (left axis)
    3. Plot of the score critic loss across steps
    """

    plot_n_generated_samples: int
    plot_n_data_samples_per_mode: int
    keep_loss_last_n: int


class MonitorConfig(BaseConfig):
    constraint_monitor_config: ConstraintMonitorConfig
    critic_monitor_config: CriticMonitorConfig
    integrated_monitor_config: IntegratedMonitorConfig
    log_every_n_steps_constraint_pretrain: int
    log_every_n_steps_critic_pretrain: int
    log_every_n_steps_integrated: int

In [5]:
monitor_config = MonitorConfig(
    constraint_monitor_config=ConstraintMonitorConfig(
        plot_n_sample_pairs=1_000,
        plot_n_data_latents_per_mode=5_000,
    ),
    critic_monitor_config=CriticMonitorConfig(
        sample_t_values=[0.03, 0.2, 0.5],
        num_contour_lines=10,
        plot_n_data_latents_per_mode=5_000,
        plot_n_vectors_per_mode=50,
    ),
    integrated_monitor_config=IntegratedMonitorConfig(
        plot_n_generated_samples=3_000,
        plot_n_data_samples_per_mode=500,
        keep_loss_last_n=4_000,
    ),
    log_every_n_steps_constraint_pretrain=1000,
    log_every_n_steps_critic_pretrain=1000,
    log_every_n_steps_integrated=4000,
)
monitor_config.visualize()

In [6]:
MODE_COLORS = (
    "#1f77b4",
    "#ff7f0e",
    "#2ca02c",
    "#d62728",
    "#9467bd",
    "#8c564b",
)


@contextlib.contextmanager
def runtime_precision_context():
    with contextlib.ExitStack() as stack:
        if device_type == "cuda":
            stack.enter_context(torch.cuda.device(device))
        stack.enter_context(torch.device(str(device)))
        stack.enter_context(torch.autocast(device_type=device_type, dtype=torch.bfloat16))
        yield


def should_log_monitor(
    *,
    step: int,
    total_steps: int,
    every_n_steps: int,
) -> bool:
    return step == 1 or step % every_n_steps == 0 or step == total_steps


def format_metrics_summary(
    metrics: dict[str, object],
) -> str:
    return ", ".join(
        f"{key}={value:.4f}"
        for key, value in metrics.items()
        if isinstance(value, float)
    )


def sample_monitor_batch(
    *,
    batch_size_per_mode: int,
) -> tuple[torch.Tensor, torch.Tensor]:
    batches = []
    labels = []
    for mode_id in range(runtime_data_config.num_modes):
        batches.append(
            runtime_data_config.sample_class(
                mode_id=mode_id,
                batch_size=batch_size_per_mode,
            )
        )
        labels.append(
            torch.full(
                (batch_size_per_mode,),
                fill_value=mode_id,
                device=device,
                dtype=torch.long,
            )
        )
    return torch.cat(batches, dim=0), torch.cat(labels, dim=0)


def sample_monitor_pairs_batch(
    *,
    total_batch_size: int,
) -> tuple[torch.Tensor, torch.Tensor]:
    batch_size_per_mode = max(
        1,
        (total_batch_size + runtime_data_config.num_modes - 1)
        // runtime_data_config.num_modes,
    )
    samples, labels = sample_monitor_batch(
        batch_size_per_mode=batch_size_per_mode,
    )
    return samples[:total_batch_size], labels[:total_batch_size]


def write_plot_artifacts(
    *,
    figure,
    step: int,
    plot_type: str,
    stage: str,
    artifact_subdir: str = "",
) -> None:
    output_folder = training_config.folder / stage / str(step)
    if artifact_subdir:
        output_folder = output_folder / artifact_subdir
    output_folder.mkdir(parents=True, exist_ok=True)
    figure.write_html(output_folder / f"{plot_type}.html")
    figure.write_image(output_folder / f"{plot_type}.png")


def flatten_latents(
    points: torch.Tensor,
) -> torch.Tensor:
    return points.reshape(points.shape[0], -1).to(dtype=torch.float32)


def orient_projection_basis(
    basis: torch.Tensor,
) -> torch.Tensor:
    oriented_basis = basis.clone()
    for component_index in range(oriented_basis.shape[1]):
        component = oriented_basis[:, component_index]
        dominant_coordinate = int(component.abs().argmax().item())
        if component[dominant_coordinate] < 0.0:
            oriented_basis[:, component_index] = -component
    return oriented_basis


def fit_latent_pca_projection(
    *,
    reference_points: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    flat_reference_points = flatten_latents(reference_points)
    projection_center = flat_reference_points.mean(dim=0, keepdim=True)
    centered_points = flat_reference_points - projection_center

    if flat_reference_points.shape[-1] == 1:
        projection_basis = torch.zeros(
            (1, 2),
            device=flat_reference_points.device,
            dtype=flat_reference_points.dtype,
        )
        projection_basis[0, 0] = 1.0
        return projection_center, projection_basis

    _, _, right_singular_vectors = torch.linalg.svd(
        centered_points,
        full_matrices=False,
    )
    projection_basis = right_singular_vectors[:2].transpose(0, 1).contiguous()
    if projection_basis.shape[1] == 1:
        projection_basis = torch.cat(
            [
                projection_basis,
                torch.zeros(
                    (projection_basis.shape[0], 1),
                    device=projection_basis.device,
                    dtype=projection_basis.dtype,
                ),
            ],
            dim=1,
        )
    return projection_center, orient_projection_basis(projection_basis)


def project_latents_to_pca_plane(
    *,
    reference_points: torch.Tensor,
    points: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    projection_center, projection_basis = fit_latent_pca_projection(
        reference_points=reference_points,
    )
    flat_points = flatten_latents(points)
    centered_points = flat_points - projection_center
    projected_points = centered_points @ projection_basis
    reconstructed_points = projection_center + projected_points @ projection_basis.transpose(0, 1)
    off_plane_norm = (flat_points - reconstructed_points).norm(dim=-1)
    return projected_points, off_plane_norm


def project_latent_vectors_to_pca_plane(
    *,
    reference_points: torch.Tensor,
    vectors: torch.Tensor,
) -> torch.Tensor:
    _, projection_basis = fit_latent_pca_projection(reference_points=reference_points)
    return flatten_latents(vectors) @ projection_basis


def latent_square_limits(
    points: torch.Tensor,
    *,
    padding: float = 0.18,
) -> tuple[float, float, float, float]:
    mins = points.min(dim=0).values
    maxs = points.max(dim=0).values
    center = 0.5 * (mins + maxs)
    radius = 0.5 * float((maxs - mins).max().item())
    radius = max(radius * (1.0 + padding), 1.0)
    return (
        float(center[0] - radius),
        float(center[0] + radius),
        float(center[1] - radius),
        float(center[1] + radius),
    )


def build_latent_grid(
    *,
    points: torch.Tensor,
    resolution: int,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    projected_points, _ = project_latents_to_pca_plane(
        reference_points=points,
        points=points,
    )
    x_min, x_max, y_min, y_max = latent_square_limits(projected_points)
    xs = torch.linspace(
        x_min,
        x_max,
        resolution,
        device=projected_points.device,
        dtype=projected_points.dtype,
    )
    ys = torch.linspace(
        y_min,
        y_max,
        resolution,
        device=projected_points.device,
        dtype=projected_points.dtype,
    )
    grid_y, grid_x = torch.meshgrid(ys, xs, indexing="ij")
    projected_grid_points = torch.stack(
        [grid_x.reshape(-1), grid_y.reshape(-1)],
        dim=-1,
    )
    projection_center, projection_basis = fit_latent_pca_projection(
        reference_points=points,
    )
    grid_points = projection_center + projected_grid_points @ projection_basis.transpose(0, 1)
    return grid_points.to(device=points.device, dtype=points.dtype), xs, ys


def vector_display_length(
    points: torch.Tensor,
    *,
    fraction: float = 0.02,
) -> float:
    x_min, x_max, y_min, y_max = latent_square_limits(points, padding=0.0)
    return fraction * max(x_max - x_min, y_max - y_min)


def normalize_vectors(
    *,
    vectors: torch.Tensor,
    display_length: float,
) -> torch.Tensor:
    magnitudes = vectors.norm(dim=-1, keepdim=True)
    return display_length * vectors / magnitudes.clamp_min(1e-6)


def add_mode_scatter_traces(
    *,
    figure: go.Figure,
    points: torch.Tensor,
    labels: torch.Tensor,
    size: float,
    opacity: float,
    showlegend: bool,
    line_width: float = 0.0,
) -> None:
    points_float = points.to(dtype=torch.float32)
    labels_long = labels.to(dtype=torch.long)
    for mode_id in range(runtime_data_config.num_modes):
        mask = labels_long == mode_id
        if not mask.any():
            continue
        figure.add_trace(
            go.Scatter(
                x=points_float[mask, 0].tolist(),
                y=points_float[mask, 1].tolist(),
                mode="markers",
                name=f"mode {mode_id}",
                showlegend=showlegend,
                marker={
                    "size": size,
                    "opacity": opacity,
                    "color": MODE_COLORS[mode_id % len(MODE_COLORS)],
                    "line": {
                        "width": line_width,
                        "color": "rgba(0, 0, 0, 0.30)",
                    },
                },
            )
        )


def add_mode_latent_annotation_traces(
    *,
    figure: go.Figure,
    points: torch.Tensor,
    labels: torch.Tensor,
    latent_norms: torch.Tensor,
    off_plane_norms: torch.Tensor,
    size: float,
    opacity: float,
    showlegend: bool,
    extra_norm_name: str = "",
    extra_norms=None,
) -> None:
    points_float = points.to(dtype=torch.float32)
    labels_long = labels.to(dtype=torch.long)
    latent_norms_float = latent_norms.to(dtype=torch.float32)
    off_plane_norms_float = off_plane_norms.to(dtype=torch.float32)
    extra_norms_float = None
    if extra_norms is not None:
        extra_norms_float = extra_norms.to(dtype=torch.float32)

    for mode_id in range(runtime_data_config.num_modes):
        mask = labels_long == mode_id
        if not mask.any():
            continue

        if extra_norms_float is None:
            customdata = torch.stack(
                [
                    labels_long[mask].to(dtype=torch.float32),
                    latent_norms_float[mask],
                    off_plane_norms_float[mask],
                ],
                dim=-1,
            ).tolist()
            hovertemplate = (
                "label=%{customdata[0]:.0f}<br>"
                "pc1=%{x:.3f}<br>"
                "pc2=%{y:.3f}<br>"
                "|latent|=%{customdata[1]:.4f}<br>"
                "off_pca_norm=%{customdata[2]:.4f}<extra></extra>"
            )
        else:
            customdata = torch.stack(
                [
                    labels_long[mask].to(dtype=torch.float32),
                    latent_norms_float[mask],
                    off_plane_norms_float[mask],
                    extra_norms_float[mask],
                ],
                dim=-1,
            ).tolist()
            hovertemplate = (
                "label=%{customdata[0]:.0f}<br>"
                "pc1=%{x:.3f}<br>"
                "pc2=%{y:.3f}<br>"
                "|latent|=%{customdata[1]:.4f}<br>"
                "off_pca_norm=%{customdata[2]:.4f}<br>"
                + extra_norm_name
                + "=%{customdata[3]:.4f}<extra></extra>"
            )

        figure.add_trace(
            go.Scatter(
                x=points_float[mask, 0].tolist(),
                y=points_float[mask, 1].tolist(),
                mode="markers",
                name=f"mode {mode_id}",
                showlegend=showlegend,
                marker={
                    "size": size,
                    "opacity": opacity,
                    "color": MODE_COLORS[mode_id % len(MODE_COLORS)],
                    "line": {
                        "width": 0.0,
                    },
                },
                customdata=customdata,
                hovertemplate=hovertemplate,
            )
        )


def add_mode_quiver_traces(
    *,
    figure: go.Figure,
    points: torch.Tensor,
    vectors: torch.Tensor,
    labels: torch.Tensor,
) -> None:
    points_float = points.to(dtype=torch.float32)
    vectors_float = vectors.to(dtype=torch.float32)
    labels_long = labels.to(dtype=torch.long)
    for mode_id in range(runtime_data_config.num_modes):
        mask = labels_long == mode_id
        if not mask.any():
            continue
        quiver = ff.create_quiver(
            x=points_float[mask, 0].tolist(),
            y=points_float[mask, 1].tolist(),
            u=vectors_float[mask, 0].tolist(),
            v=vectors_float[mask, 1].tolist(),
            scale=1.0,
            arrow_scale=0.25,
            line_color=MODE_COLORS[mode_id % len(MODE_COLORS)],
            name=f"mode {mode_id}",
        )
        for trace in quiver.data:
            trace.showlegend = False
            trace.hoverinfo = "skip"
            trace.line.width = 1.2
            figure.add_trace(trace)


def add_mode_quiver_hover_traces(
    *,
    figure: go.Figure,
    points: torch.Tensor,
    display_vectors: torch.Tensor,
    vector_norms: torch.Tensor,
    labels: torch.Tensor,
) -> None:
    points_float = points.to(dtype=torch.float32)
    display_vectors_float = display_vectors.to(dtype=torch.float32)
    vector_norms_float = vector_norms.to(dtype=torch.float32)
    labels_long = labels.to(dtype=torch.long)
    hover_points = points_float + 0.5 * display_vectors_float
    for mode_id in range(runtime_data_config.num_modes):
        mask = labels_long == mode_id
        if not mask.any():
            continue
        figure.add_trace(
            go.Scatter(
                x=hover_points[mask, 0].tolist(),
                y=hover_points[mask, 1].tolist(),
                mode="markers",
                marker={
                    "size": 10,
                    "color": MODE_COLORS[mode_id % len(MODE_COLORS)],
                    "opacity": 0.18,
                    "line": {"width": 0},
                },
                customdata=vector_norms_float[mask].unsqueeze(-1).tolist(),
                hovertemplate="arrow_norm=%{customdata[0]:.4f}<extra></extra>",
                showlegend=False,
                name=f"mode {mode_id} hover",
            )
        )


def sample_transport_times(
    *,
    batch_shape: tuple[int, ...],
) -> torch.Tensor:
    return transport_t_min + (transport_t_max - transport_t_min) * torch.rand(
        *batch_shape,
        device=device,
        dtype=torch.float32,
    )


def sample_stratified_transport_times(
    *,
    batch_size: int,
    num_time_samples: int,
) -> torch.Tensor:
    bin_edges = torch.linspace(
        transport_t_min,
        transport_t_max,
        num_time_samples + 1,
        device=device,
        dtype=torch.float32,
    )
    bin_starts = bin_edges[:-1]
    bin_widths = bin_edges[1:] - bin_edges[:-1]
    return bin_starts.unsqueeze(0) + bin_widths.unsqueeze(0) * torch.rand(
        batch_size,
        num_time_samples,
        device=device,
        dtype=torch.float32,
    )


def critic_score_from_noise_prediction(
    *,
    predicted_noise: torch.Tensor,
    t: torch.Tensor,
) -> torch.Tensor:
    return -predicted_noise / t.unsqueeze(-1)


def critic_spectrum_t_values() -> list[float]:
    return torch.linspace(
        transport_t_min,
        transport_t_max,
        19,
        dtype=torch.float32,
    ).tolist()


def sample_critic_snapshot(
    *,
    clean_latents: torch.Tensor,
    t_value: float,
) -> tuple[torch.Tensor, torch.Tensor]:
    t = torch.full(
        (clean_latents.shape[0],),
        float(t_value),
        device=clean_latents.device,
        dtype=torch.float32,
    )
    noise = torch.randn_like(clean_latents)
    noised_latents = (
        (1.0 - t).unsqueeze(-1) * clean_latents + t.unsqueeze(-1) * noise
    )
    predicted_noise = critic(noised_latents, t).float()
    data_score = critic_score_from_noise_prediction(
        predicted_noise=predicted_noise,
        t=t,
    )
    return noised_latents.float(), data_score.float()


def estimate_clean_transport_field(
    *,
    clean_latents: torch.Tensor,
    t_values: list[float],
) -> torch.Tensor:
    if not t_values:
        raise ValueError("t_values must be non-empty")

    transport_field = torch.zeros_like(clean_latents, dtype=torch.float32)
    for t_value in t_values:
        t = torch.full(
            (clean_latents.shape[0],),
            float(t_value),
            device=clean_latents.device,
            dtype=torch.float32,
        )
        pullback_weight = transport_config.kl_weight_schedule.pullback_weight(
            t.float(),
        ).unsqueeze(-1)
        noise = torch.randn_like(clean_latents)

        def evaluate_with_noise(sampled_noise: torch.Tensor) -> torch.Tensor:
            noised_latents = (
                (1.0 - t).unsqueeze(-1) * clean_latents
                + t.unsqueeze(-1) * sampled_noise
            )
            predicted_noise = critic(noised_latents, t).float()
            prior_score = prior_config.analytic_score(
                noised_latents.float(),
                t.float(),
            ).float()
            return pullback_weight * (prior_score + predicted_noise / t.unsqueeze(-1))

        transport_terms = evaluate_with_noise(noise)
        if transport_config.antipodal_estimate:
            transport_terms = 0.5 * (
                transport_terms + evaluate_with_noise(-noise)
            )
        transport_field = transport_field + transport_terms
    return transport_field / len(t_values)


def estimate_critic_loss_spectrum(
    *,
    clean_latents: torch.Tensor,
    t_values: list[float],
) -> pl.DataFrame:
    rows = []
    for t_value in t_values:
        t = torch.full(
            (clean_latents.shape[0],),
            float(t_value),
            device=clean_latents.device,
            dtype=torch.float32,
        )
        noise = torch.randn_like(clean_latents)

        def mse_with_noise(sampled_noise: torch.Tensor) -> torch.Tensor:
            noised_latents = (
                (1.0 - t).unsqueeze(-1) * clean_latents
                + t.unsqueeze(-1) * sampled_noise
            )
            predicted_noise = critic(noised_latents, t).float()
            return (predicted_noise - sampled_noise.float()).square().mean()

        noise_prediction_mse = mse_with_noise(noise)
        if transport_config.antipodal_estimate:
            noise_prediction_mse = 0.5 * (
                noise_prediction_mse + mse_with_noise(-noise)
            )

        rows.append(
            {
                "t": float(t_value),
                "noise_prediction_mse": float(noise_prediction_mse.item()),
            }
        )
    return pl.DataFrame(rows)


def plot_critic_score_snapshot(
    *,
    reference_latents: torch.Tensor,
    cloud_latents: torch.Tensor,
    cloud_labels: torch.Tensor,
    arrow_latents: torch.Tensor,
    arrow_labels: torch.Tensor,
    data_score: torch.Tensor,
    t_value: float,
) -> go.Figure:
    projected_cloud_latents, cloud_off_plane_norm = project_latents_to_pca_plane(
        reference_points=reference_latents,
        points=cloud_latents,
    )
    projected_arrow_latents, arrow_off_plane_norm = project_latents_to_pca_plane(
        reference_points=reference_latents,
        points=arrow_latents,
    )
    projected_data_score = project_latent_vectors_to_pca_plane(
        reference_points=reference_latents,
        vectors=data_score,
    )
    cloud_latent_norm = flatten_latents(cloud_latents).norm(dim=-1)
    arrow_latent_norm = flatten_latents(arrow_latents).norm(dim=-1)
    data_score_norm = flatten_latents(data_score).norm(dim=-1)

    figure = go.Figure()
    add_mode_latent_annotation_traces(
        figure=figure,
        points=projected_cloud_latents,
        labels=cloud_labels,
        latent_norms=cloud_latent_norm,
        off_plane_norms=cloud_off_plane_norm,
        size=5,
        opacity=0.28,
        showlegend=True,
    )
    add_mode_latent_annotation_traces(
        figure=figure,
        points=projected_arrow_latents,
        labels=arrow_labels,
        latent_norms=arrow_latent_norm,
        off_plane_norms=arrow_off_plane_norm,
        size=5,
        opacity=0.38,
        showlegend=False,
        extra_norm_name="score_norm",
        extra_norms=data_score_norm,
    )
    add_mode_quiver_traces(
        figure=figure,
        points=projected_arrow_latents,
        vectors=normalize_vectors(
            vectors=projected_data_score,
            display_length=vector_display_length(projected_cloud_latents),
        ),
        labels=arrow_labels,
    )
    x_min, x_max, y_min, y_max = latent_square_limits(projected_cloud_latents)
    figure.update_layout(
        title=f"Critic score snapshot at t={t_value:.2f} (PCA view)",
        template="plotly_white",
        width=850,
        height=850,
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0.0},
        xaxis={"range": [x_min, x_max], "title": "pc1"},
        yaxis={"range": [y_min, y_max], "title": "pc2", "scaleanchor": "x", "scaleratio": 1.0},
    )
    return figure


def plot_transport_field(
    *,
    reference_latents: torch.Tensor,
    cloud_latents: torch.Tensor,
    cloud_labels: torch.Tensor,
    arrow_latents: torch.Tensor,
    arrow_labels: torch.Tensor,
    transport_field: torch.Tensor,
    grid_xs: torch.Tensor,
    grid_ys: torch.Tensor,
    grid_transport_projection: torch.Tensor,
    num_contour_lines: int,
) -> go.Figure:
    projected_cloud_latents, cloud_off_plane_norm = project_latents_to_pca_plane(
        reference_points=reference_latents,
        points=cloud_latents,
    )
    projected_arrow_latents, arrow_off_plane_norm = project_latents_to_pca_plane(
        reference_points=reference_latents,
        points=arrow_latents,
    )
    projected_transport_field = project_latent_vectors_to_pca_plane(
        reference_points=reference_latents,
        vectors=transport_field,
    )
    cloud_latent_norm = flatten_latents(cloud_latents).norm(dim=-1)
    arrow_latent_norm = flatten_latents(arrow_latents).norm(dim=-1)
    transport_field_norm = flatten_latents(transport_field).norm(dim=-1)

    figure = go.Figure()
    figure.add_trace(
        go.Contour(
            x=grid_xs.to(dtype=torch.float32).tolist(),
            y=grid_ys.to(dtype=torch.float32).tolist(),
            z=grid_transport_projection.to(dtype=torch.float32).numpy(),
            contours={"coloring": "lines", "showlabels": True},
            line={"width": 1.1, "color": "rgba(70, 70, 70, 0.55)"},
            ncontours=num_contour_lines,
            showscale=False,
            name="transport norm contours",
            hoverinfo="skip",
        )
    )
    add_mode_latent_annotation_traces(
        figure=figure,
        points=projected_cloud_latents,
        labels=cloud_labels,
        latent_norms=cloud_latent_norm,
        off_plane_norms=cloud_off_plane_norm,
        size=5,
        opacity=0.28,
        showlegend=True,
    )
    add_mode_latent_annotation_traces(
        figure=figure,
        points=projected_arrow_latents,
        labels=arrow_labels,
        latent_norms=arrow_latent_norm,
        off_plane_norms=arrow_off_plane_norm,
        size=5,
        opacity=0.38,
        showlegend=False,
        extra_norm_name="transport_norm",
        extra_norms=transport_field_norm,
    )
    display_vectors = normalize_vectors(
        vectors=projected_transport_field,
        display_length=vector_display_length(projected_cloud_latents),
    )
    add_mode_quiver_traces(
        figure=figure,
        points=projected_arrow_latents,
        vectors=display_vectors,
        labels=arrow_labels,
    )
    add_mode_quiver_hover_traces(
        figure=figure,
        points=projected_arrow_latents,
        display_vectors=display_vectors,
        vector_norms=transport_field_norm,
        labels=arrow_labels,
    )
    x_min, x_max, y_min, y_max = latent_square_limits(projected_cloud_latents)
    figure.update_layout(
        title="Noise-averaged clean-latent transport field (PCA view)",
        template="plotly_white",
        width=850,
        height=850,
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0.0},
        xaxis={"range": [x_min, x_max], "title": "pc1"},
        yaxis={"range": [y_min, y_max], "title": "pc2", "scaleanchor": "x", "scaleratio": 1.0},
    )
    return figure


def plot_critic_loss_spectrum(
    *,
    loss_spectrum: pl.DataFrame,
) -> go.Figure:
    figure = go.Figure()
    figure.add_trace(
        go.Scatter(
            x=loss_spectrum["t"].to_list(),
            y=loss_spectrum["noise_prediction_mse"].to_list(),
            mode="lines+markers",
            line={"color": "#1f77b4", "width": 2.0},
            marker={"size": 7},
            name="noise prediction mse",
        )
    )
    figure.update_layout(
        title="Critic loss spectrum",
        template="plotly_white",
        width=900,
        height=500,
        xaxis={},
        yaxis={},
    )
    return figure


def plot_generated_samples(
    *,
    data_samples: torch.Tensor,
    data_labels: torch.Tensor,
    generated_samples: torch.Tensor,
) -> go.Figure:
    if data_samples.shape[-1] != 2 or generated_samples.shape[-1] != 2:
        raise ValueError("Generated sample monitor plots require 2D projected samples")

    figure = go.Figure()
    add_mode_scatter_traces(
        figure=figure,
        points=data_samples,
        labels=data_labels,
        size=5,
        opacity=0.28,
        showlegend=True,
    )
    figure.add_trace(
        go.Scatter(
            x=generated_samples[:, 0].to(dtype=torch.float32).tolist(),
            y=generated_samples[:, 1].to(dtype=torch.float32).tolist(),
            mode="markers",
            marker={
                "size": 6,
                "opacity": 0.55,
                "color": "rgba(20, 20, 20, 0.80)",
                "line": {"width": 0.0},
            },
            name="generated samples",
        )
    )
    all_points = torch.cat([data_samples, generated_samples], dim=0)
    x_min, x_max, y_min, y_max = latent_square_limits(all_points)
    figure.update_layout(
        title="Generated samples vs data",
        template="plotly_white",
        width=850,
        height=850,
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0.0},
        xaxis={"range": [x_min, x_max]},
        yaxis={"range": [y_min, y_max], "scaleanchor": "x", "scaleratio": 1.0},
    )
    return figure


def history_plot_values(
    *,
    history: pl.DataFrame,
    column: str,
) -> list[float | None]:
    values = []
    for value in history[column].to_list():
        if isinstance(value, float) and value != value:
            values.append(None)
        else:
            values.append(float(value))
    return values


def plot_constraint_history(
    *,
    history: pl.DataFrame,
) -> go.Figure:
    steps = history["step"].to_list()
    figure = make_subplots(specs=[[{"secondary_y": True}]])
    figure.add_trace(
        go.Scatter(
            x=steps,
            y=history_plot_values(history=history, column="data_cycle_loss"),
            mode="lines",
            line={"color": "#1f77b4", "width": 2.0},
            name="data cycle loss",
        ),
        secondary_y=False,
    )
    figure.add_trace(
        go.Scatter(
            x=steps,
            y=history_plot_values(history=history, column="prior_cycle_loss"),
            mode="lines",
            line={"color": "#ff7f0e", "width": 2.0},
            name="prior cycle loss",
        ),
        secondary_y=False,
    )
    figure.add_trace(
        go.Scatter(
            x=steps,
            y=history_plot_values(history=history, column="data_dual"),
            mode="lines",
            line={"color": "#2ca02c", "width": 2.0},
            name="data dual",
        ),
        secondary_y=True,
    )
    figure.add_trace(
        go.Scatter(
            x=steps,
            y=history_plot_values(history=history, column="prior_dual"),
            mode="lines",
            line={"color": "#d62728", "width": 2.0},
            name="prior dual",
        ),
        secondary_y=True,
    )
    figure.update_layout(
        title="Constraint losses and dual variables",
        template="plotly_white",
        width=1100,
        height=500,
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0.0},
    )
    figure.update_xaxes(title_text="step")
    figure.update_yaxes(title_text="loss", secondary_y=False)
    figure.update_yaxes(title_text="dual", secondary_y=True)
    return figure


def plot_transport_history(
    *,
    history: pl.DataFrame,
) -> go.Figure:
    steps = history["step"].to_list()
    figure = make_subplots(specs=[[{"secondary_y": True}]])
    figure.add_trace(
        go.Scatter(
            x=steps,
            y=history_plot_values(history=history, column="critic_loss"),
            mode="lines",
            line={"color": "#1f77b4", "width": 2.0},
            name="critic loss",
            connectgaps=True,
        ),
        secondary_y=False,
    )
    figure.add_trace(
        go.Scatter(
            x=steps,
            y=history_plot_values(history=history, column="transport_field_norm"),
            mode="lines+markers",
            line={"color": "#ff7f0e", "width": 3.0, "dash": "dash"},
            marker={"size": 4, "color": "#ff7f0e"},
            name="transport field norm",
            connectgaps=True,
        ),
        secondary_y=True,
    )
    figure.update_layout(
        title="Transport field norm and critic loss",
        template="plotly_white",
        width=1100,
        height=500,
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0.0},
    )
    figure.update_xaxes(title_text="step")
    figure.update_yaxes(title_text="critic loss", secondary_y=False)
    figure.update_yaxes(title_text="transport field norm", secondary_y=True)
    return figure


def plot_generated_log_likelihood_history(
    *,
    history: pl.DataFrame,
) -> go.Figure:
    figure = go.Figure()
    figure.add_trace(
        go.Scatter(
            x=history["step"].to_list(),
            y=history_plot_values(history=history, column="avg_generated_log_likelihood"),
            mode="lines+markers",
            line={"color": "#2ca02c", "width": 2.5},
            marker={"size": 4, "color": "#2ca02c"},
            name="avg generated log likelihood",
            connectgaps=True,
        )
    )
    figure.update_layout(
        title="Generated data log likelihood",
        template="plotly_white",
        width=1100,
        height=500,
        xaxis={"title": "step"},
        yaxis={"title": "avg generated log likelihood"},
    )
    return figure


def write_constraint_monitor_artifacts(
    *,
    step: int,
    stage: str,
    plot_prefix: str,
    pair_projected_samples: torch.Tensor,
    pair_projected_reconstructions: torch.Tensor,
    pair_manifold_deviation: torch.Tensor,
    pair_labels: torch.Tensor,
    latent_values: torch.Tensor,
    latent_off_manifold_norm: torch.Tensor,
    latent_labels: torch.Tensor,
) -> None:
    sample_pairs_figure = plot_sample_pairs(
        samples=pair_projected_samples,
        pairs=pair_projected_reconstructions,
        manifold_deviation=pair_manifold_deviation,
        labels=pair_labels,
    )
    write_plot_artifacts(
        figure=sample_pairs_figure,
        step=step,
        plot_type=f"{plot_prefix}sample_pairs",
        stage=stage,
    )

    latents_figure = plot_latents(
        latents=latent_values,
        off_manifold_norm=latent_off_manifold_norm,
        labels=latent_labels,
    )
    write_plot_artifacts(
        figure=latents_figure,
        step=step,
        plot_type=f"{plot_prefix}latents",
        stage=stage,
    )


def write_critic_monitor_artifacts(
    *,
    step: int,
    stage: str,
    plot_prefix: str,
    dense_clean_latents: torch.Tensor,
    dense_labels: torch.Tensor,
    vector_clean_latents: torch.Tensor,
    vector_labels: torch.Tensor,
    score_snapshots: list[tuple[float, torch.Tensor, torch.Tensor, torch.Tensor]],
    transport_field: torch.Tensor,
    transport_grid_xs: torch.Tensor,
    transport_grid_ys: torch.Tensor,
    transport_grid_projection: torch.Tensor,
    loss_spectrum: pl.DataFrame,
    num_contour_lines: int,
    score_snapshot_subdir: str = "",
) -> None:
    for t_value, cloud_latents, arrow_latents, data_score in score_snapshots:
        score_figure = plot_critic_score_snapshot(
            reference_latents=dense_clean_latents,
            cloud_latents=cloud_latents,
            cloud_labels=dense_labels,
            arrow_latents=arrow_latents,
            arrow_labels=vector_labels,
            data_score=data_score,
            t_value=t_value,
        )
        write_plot_artifacts(
            figure=score_figure,
            step=step,
            plot_type=f"{plot_prefix}score_snapshot_t_{t_value:.2f}".replace(".", "p"),
            stage=stage,
            artifact_subdir=score_snapshot_subdir,
        )

    transport_figure = plot_transport_field(
        reference_latents=dense_clean_latents,
        cloud_latents=dense_clean_latents,
        cloud_labels=dense_labels,
        arrow_latents=vector_clean_latents,
        arrow_labels=vector_labels,
        transport_field=transport_field,
        grid_xs=transport_grid_xs,
        grid_ys=transport_grid_ys,
        grid_transport_projection=transport_grid_projection,
        num_contour_lines=num_contour_lines,
    )
    write_plot_artifacts(
        figure=transport_figure,
        step=step,
        plot_type=f"{plot_prefix}transport",
        stage=stage,
    )

    loss_spectrum_figure = plot_critic_loss_spectrum(
        loss_spectrum=loss_spectrum,
    )
    write_plot_artifacts(
        figure=loss_spectrum_figure,
        step=step,
        plot_type=f"{plot_prefix}loss_spectrum",
        stage=stage,
    )


def write_integrated_monitor_artifacts(
    *,
    step: int,
    stage: str,
    generated_samples: torch.Tensor,
    generated_data_samples: torch.Tensor,
    generated_data_labels: torch.Tensor,
    recent_history: pl.DataFrame,
) -> None:
    generated_figure = plot_generated_samples(
        data_samples=generated_data_samples,
        data_labels=generated_data_labels,
        generated_samples=generated_samples,
    )
    write_plot_artifacts(
        figure=generated_figure,
        step=step,
        plot_type="generated_samples",
        stage=stage,
    )

    constraint_history_figure = plot_constraint_history(
        history=recent_history,
    )
    write_plot_artifacts(
        figure=constraint_history_figure,
        step=step,
        plot_type="constraint_history",
        stage=stage,
    )

    transport_history_figure = plot_transport_history(
        history=recent_history,
    )
    write_plot_artifacts(
        figure=transport_history_figure,
        step=step,
        plot_type="transport_history",
        stage=stage,
    )

    generated_log_likelihood_figure = plot_generated_log_likelihood_history(
        history=recent_history,
    )
    write_plot_artifacts(
        figure=generated_log_likelihood_figure,
        step=step,
        plot_type="generated_log_likelihood",
        stage=stage,
    )



## Pretrain

In [7]:
pretrain_history = []
encoder.train()
decoder.train()
critic.eval()

pretrain_progress = tqdm(
    range(1, chart_transport_config.scheduling_config.pretrain_chart_n_steps + 1),
    desc="pretrain",
)
for step in pretrain_progress:
    optimizer.zero_grad(set_to_none=True)

    with runtime_precision_context():
        data_batch = runtime_data_config.sample_unconditional(
            batch_size=training_config.train_batch_size,
        )
        prior_batch = prior_config.sample(
            batch_size=training_config.train_batch_size,
        ).to(device=device, dtype=torch.float32)

        data_latents = encoder(data_batch)
        data_reconstruction = decoder(data_latents)
        prior_reconstruction = decoder(prior_batch)
        prior_latents = encoder(prior_reconstruction)

        data_cycle_loss = F.huber_loss(
            data_reconstruction,
            data_batch,
            delta=constraint_config.huber_delta,
            reduction="mean",
        )
        prior_cycle_loss = F.huber_loss(
            prior_latents,
            prior_batch,
            delta=constraint_config.huber_delta,
            reduction="mean",
        )
        constraint_loss = data_cycle_loss + prior_cycle_loss

        zero_mean_loss = F.huber_loss(
            data_latents.mean(),
            torch.zeros((), device=device, dtype=data_latents.dtype),
            delta=1.0,
            reduction="mean",
        )
        latent_norms = data_latents.square().sum(dim=-1).sqrt()
        latent_norm_loss = F.huber_loss(
            latent_norms,
            torch.zeros_like(latent_norms),
            delta=pretrain_config.latent_norm_delta,
            reduction="mean",
        )
        chart_loss = constraint_loss
        chart_loss = chart_loss + pretrain_config.zero_mean_weight * zero_mean_loss
        chart_loss = chart_loss + pretrain_config.latent_norm_weight * latent_norm_loss

    fabric.backward(chart_loss)
    fabric.clip_gradients(
        chart_transport_model,
        optimizer,
        max_norm=architecture_config.grad_clip_norm,
    )
    optimizer.step()

    metrics = {
        "stage": "pretrain",
        "step": step,
        "chart_loss": chart_loss.detach().item(),
        "data_cycle_loss": data_cycle_loss.detach().item(),
        "prior_cycle_loss": prior_cycle_loss.detach().item(),
        "zero_mean_loss": zero_mean_loss.detach().item(),
        "latent_norm_loss": latent_norm_loss.detach().item(),
    }
    pretrain_history.append(metrics)
    pretrain_progress.set_postfix(
        chart_loss=f"{metrics['chart_loss']:.4f}",
        data_cycle=f"{metrics['data_cycle_loss']:.4f}",
        prior_cycle=f"{metrics['prior_cycle_loss']:.4f}",
    )

    if should_log_monitor(
        step=step,
        total_steps=chart_transport_config.scheduling_config.pretrain_chart_n_steps,
        every_n_steps=monitor_config.log_every_n_steps_constraint_pretrain,
    ):
        fabric.print(
            f"[pretrain] step {step}/{chart_transport_config.scheduling_config.pretrain_chart_n_steps}: {format_metrics_summary(metrics)}"
        )

        encoder_was_training = encoder.training
        decoder_was_training = decoder.training
        encoder.eval()
        decoder.eval()

        with torch.no_grad():
            with runtime_precision_context():
                pair_samples, pair_labels = sample_monitor_pairs_batch(
                    total_batch_size=monitor_config.constraint_monitor_config.plot_n_sample_pairs,
                )
                pair_latents = encoder(pair_samples)
                pair_reconstructions = decoder(pair_latents)

                latent_samples, latent_labels = sample_monitor_batch(
                    batch_size_per_mode=monitor_config.constraint_monitor_config.plot_n_data_latents_per_mode,
                )
                latent_values = encoder(latent_samples)

            pair_samples_cpu = pair_samples.detach().cpu().float()
            pair_reconstructions_cpu = pair_reconstructions.detach().cpu().float()
            pair_labels_cpu = pair_labels.detach().cpu().long()
            pair_projected_samples, _, _ = data_config.decompose_projection(
                pair_samples_cpu,
            )
            pair_projected_reconstructions, _, _ = data_config.decompose_projection(
                pair_reconstructions_cpu,
            )
            pair_manifold_deviation = (
                (pair_reconstructions - pair_samples)
                .reshape(pair_samples.shape[0], -1)
                .norm(dim=-1)
                .detach()
                .cpu()
                .float()
            )

            latent_samples_cpu = latent_samples.detach().cpu().float()
            latent_values_cpu = latent_values.detach().cpu().float()
            latent_labels_cpu = latent_labels.detach().cpu().long()
            _, _, latent_off_plane = data_config.decompose_projection(
                latent_samples_cpu,
            )
            latent_off_manifold_norm = latent_off_plane.norm(dim=-1).float()

        if encoder_was_training:
            encoder.train()
        if decoder_was_training:
            decoder.train()

        write_constraint_monitor_artifacts(
            step=step,
            stage="pretrain",
            plot_prefix="",
            pair_projected_samples=pair_projected_samples,
            pair_projected_reconstructions=pair_projected_reconstructions,
            pair_manifold_deviation=pair_manifold_deviation,
            pair_labels=pair_labels_cpu,
            latent_values=latent_values_cpu,
            latent_off_manifold_norm=latent_off_manifold_norm,
            latent_labels=latent_labels_cpu,
        )

pretrain_history = pl.DataFrame(pretrain_history)
display(pretrain_history.tail())


pretrain:   0%|          | 0/1000 [00:00<?, ?it/s]

[pretrain] step 1/1000: chart_loss=0.6169, data_cycle_loss=0.1017, prior_cycle_loss=0.5144, zero_mean_loss=0.0000, latent_norm_loss=0.0760


RuntimeError: The total norm of order 2.0 for gradients from `parameters` is non-finite, so it cannot be clipped. To disable this error and scale the gradients by the non-finite norm anyway, set `error_if_nonfinite=False`

## Train noise critic

In [ ]:
critic_pretrain_history = []
encoder.eval()
decoder.eval()
critic.train()

critic_pretrain_progress = tqdm(
    range(1, chart_transport_config.scheduling_config.pretrain_critic_n_steps + 1),
    desc="critic_pretrain",
)
for step in critic_pretrain_progress:
    optimizer.zero_grad(set_to_none=True)

    with runtime_precision_context():
        data_batch = runtime_data_config.sample_unconditional(
            batch_size=training_config.train_batch_size,
        )
        with torch.no_grad():
            data_latents = encoder(data_batch)

        t = sample_transport_times(
            batch_shape=(data_latents.shape[0],),
        )
        eps = torch.randn_like(data_latents)
        noised_latents = (
            (1.0 - t).unsqueeze(-1) * data_latents + t.unsqueeze(-1) * eps
        )
        predicted_noise = critic(noised_latents, t)
        critic_loss = F.mse_loss(predicted_noise, eps)

    fabric.backward(critic_loss)
    fabric.clip_gradients(
        chart_transport_model,
        optimizer,
        max_norm=architecture_config.grad_clip_norm,
    )
    optimizer.step()

    metrics = {
        "stage": "critic_pretrain",
        "step": step,
        "critic_loss": critic_loss.detach().item(),
    }
    critic_pretrain_history.append(metrics)
    critic_pretrain_progress.set_postfix(
        critic_loss=f"{metrics['critic_loss']:.4f}",
    )

    if should_log_monitor(
        step=step,
        total_steps=chart_transport_config.scheduling_config.pretrain_critic_n_steps,
        every_n_steps=monitor_config.log_every_n_steps_critic_pretrain,
    ):
        fabric.print(
            f"[critic_pretrain] step {step}/{chart_transport_config.scheduling_config.pretrain_critic_n_steps}: {format_metrics_summary(metrics)}"
        )

        encoder_was_training = encoder.training
        critic_was_training = critic.training
        encoder.eval()
        critic.eval()

        with torch.no_grad():
            with runtime_precision_context():
                dense_samples, dense_labels = sample_monitor_batch(
                    batch_size_per_mode=monitor_config.critic_monitor_config.plot_n_data_latents_per_mode,
                )
                vector_samples, vector_labels = sample_monitor_batch(
                    batch_size_per_mode=monitor_config.critic_monitor_config.plot_n_vectors_per_mode,
                )
                dense_clean_latents = encoder(dense_samples).float()
                vector_clean_latents = encoder(vector_samples).float()

                score_snapshots = []
                for t_value in monitor_config.critic_monitor_config.sample_t_values:
                    cloud_latents, _ = sample_critic_snapshot(
                        clean_latents=dense_clean_latents,
                        t_value=t_value,
                    )
                    arrow_latents, data_score = sample_critic_snapshot(
                        clean_latents=vector_clean_latents,
                        t_value=t_value,
                    )
                    score_snapshots.append(
                        (
                            t_value,
                            cloud_latents.detach().cpu().float(),
                            arrow_latents.detach().cpu().float(),
                            data_score.detach().cpu().float(),
                        )
                    )

                spectrum_t_values = critic_spectrum_t_values()
                transport_field = estimate_clean_transport_field(
                    clean_latents=vector_clean_latents,
                    t_values=spectrum_t_values,
                )
                transport_grid_points, transport_grid_xs, transport_grid_ys = build_latent_grid(
                    points=dense_clean_latents,
                    resolution=31,
                )
                transport_grid_field = estimate_clean_transport_field(
                    clean_latents=transport_grid_points,
                    t_values=spectrum_t_values,
                )
                transport_grid_projection = project_latent_vectors_to_pca_plane(
                    reference_points=dense_clean_latents,
                    vectors=transport_grid_field,
                ).norm(dim=-1).reshape(
                    transport_grid_ys.shape[0],
                    transport_grid_xs.shape[0],
                )
                loss_spectrum = estimate_critic_loss_spectrum(
                    clean_latents=dense_clean_latents,
                    t_values=spectrum_t_values,
                )

            dense_clean_latents_cpu = dense_clean_latents.detach().cpu().float()
            dense_labels_cpu = dense_labels.detach().cpu().long()
            vector_clean_latents_cpu = vector_clean_latents.detach().cpu().float()
            vector_labels_cpu = vector_labels.detach().cpu().long()
            transport_field_cpu = transport_field.detach().cpu().float()
            transport_grid_xs_cpu = transport_grid_xs.detach().cpu().float()
            transport_grid_ys_cpu = transport_grid_ys.detach().cpu().float()
            transport_grid_projection_cpu = (
                transport_grid_projection.detach().cpu().float()
            )

        if encoder_was_training:
            encoder.train()
        if critic_was_training:
            critic.train()

        write_critic_monitor_artifacts(
            step=step,
            stage="critic_pretrain",
            plot_prefix="",
            dense_clean_latents=dense_clean_latents_cpu,
            dense_labels=dense_labels_cpu,
            vector_clean_latents=vector_clean_latents_cpu,
            vector_labels=vector_labels_cpu,
            score_snapshots=score_snapshots,
            transport_field=transport_field_cpu,
            transport_grid_xs=transport_grid_xs_cpu,
            transport_grid_ys=transport_grid_ys_cpu,
            transport_grid_projection=transport_grid_projection_cpu,
            loss_spectrum=loss_spectrum,
            num_contour_lines=monitor_config.critic_monitor_config.num_contour_lines,
        )

critic_pretrain_history = pl.DataFrame(critic_pretrain_history)
display(critic_pretrain_history.tail())


critic_pretrain:   0%|          | 0/1000 [00:00<?, ?it/s]

[critic_pretrain] step 1/1000: critic_loss=1.1227
[critic_pretrain] step 1000/1000: critic_loss=0.0061


stage,step,critic_loss
str,i64,f64
"""critic_pretrain""",996,0.005568
"""critic_pretrain""",997,0.005622
"""critic_pretrain""",998,0.005696
"""critic_pretrain""",999,0.006106
"""critic_pretrain""",1000,0.006097


## Integrated training

In [ ]:
integrated_history = []
critic_steps_per_chart_step = max(
    1,
    chart_transport_config.scheduling_config.update_chart_every_n_critic_steps,
)
latest_chart_metrics = {
    "chart_loss": float("nan"),
    "encoder_transport_loss": float("nan"),
    "decoder_transport_loss": float("nan"),
    "transport_field_norm": float("nan"),
    "avg_generated_log_likelihood": float("nan"),
    "data_cycle_loss": float("nan"),
    "prior_cycle_loss": float("nan"),
}

integrated_progress = tqdm(
    range(1, training_config.integrated_n_steps + 1),
    desc="integrated",
)
for step in integrated_progress:
    encoder.eval()
    decoder.eval()
    critic.train()
    optimizer.zero_grad(set_to_none=True)

    with runtime_precision_context():
        critic_data_batch = runtime_data_config.sample_unconditional(
            batch_size=training_config.train_batch_size,
        )
        with torch.no_grad():
            critic_data_latents = encoder(critic_data_batch)

        critic_t = sample_transport_times(
            batch_shape=(critic_data_latents.shape[0],),
        )
        critic_eps = torch.randn_like(critic_data_latents)
        critic_noised_latents = (
            (1.0 - critic_t).unsqueeze(-1) * critic_data_latents
            + critic_t.unsqueeze(-1) * critic_eps
        )
        critic_predicted_noise = critic(critic_noised_latents, critic_t)
        critic_loss = F.mse_loss(critic_predicted_noise, critic_eps)

    fabric.backward(critic_loss)
    fabric.clip_gradients(
        chart_transport_model,
        optimizer,
        max_norm=architecture_config.grad_clip_norm,
    )
    optimizer.step()

    metrics = {
        "stage": "integrated",
        "step": step,
        "critic_loss": critic_loss.detach().item(),
        **latest_chart_metrics,
        "data_dual": data_dual.detach().item(),
        "prior_dual": prior_dual.detach().item(),
    }

    if step == 1 or step % critic_steps_per_chart_step == 0:
        encoder.train()
        decoder.train()
        optimizer.zero_grad(set_to_none=True)

        with runtime_precision_context():
            chart_data_batch = runtime_data_config.sample_unconditional(
                batch_size=training_config.train_batch_size,
            )
            chart_prior_batch = prior_config.sample(
                batch_size=training_config.train_batch_size,
            ).to(device=device, dtype=torch.float32)

            chart_data_latents = encoder(chart_data_batch)
            chart_data_reconstruction = decoder(chart_data_latents)
            chart_prior_reconstruction = decoder(chart_prior_batch)
            chart_prior_latents = encoder(chart_prior_reconstruction)

            data_cycle_loss = F.huber_loss(
                chart_data_reconstruction,
                chart_data_batch,
                delta=constraint_config.huber_delta,
                reduction="mean",
            )
            prior_cycle_loss = F.huber_loss(
                chart_prior_latents,
                chart_prior_batch,
                delta=constraint_config.huber_delta,
                reduction="mean",
            )

            if isinstance(constraint_method, LossConstraintConfig):
                constraint_loss = (
                    constraint_method.data_loss_weight * data_cycle_loss
                    + constraint_method.prior_loss_weight * prior_cycle_loss
                )
            else:
                constraint_loss = data_cycle_loss + prior_cycle_loss
                constraint_loss = constraint_loss + data_dual * (
                    data_cycle_loss - constraint_method.data_constraint_budget
                )
                constraint_loss = constraint_loss + prior_dual * (
                    prior_cycle_loss - constraint_method.prior_constraint_budget
                )

            with torch.no_grad():
                transport_source_latents = chart_data_latents.detach()
                transport_t = sample_stratified_transport_times(
                    batch_size=transport_source_latents.shape[0],
                    num_time_samples=transport_config.num_time_samples,
                )

                transport_source_latents = transport_source_latents.unsqueeze(1).expand(
                    -1,
                    transport_config.num_time_samples,
                    -1,
                )
                transport_eps = torch.randn(
                    transport_source_latents.shape[0],
                    transport_config.num_time_samples,
                    transport_source_latents.shape[-1],
                    device=device,
                    dtype=transport_source_latents.dtype,
                )

                if transport_config.antipodal_estimate:
                    transport_t = torch.cat([transport_t, transport_t], dim=1)
                    transport_source_latents = transport_source_latents.repeat(1, 2, 1)
                    transport_eps = torch.cat([transport_eps, -transport_eps], dim=1)

                transport_noised_latents = (
                    (1.0 - transport_t).unsqueeze(-1) * transport_source_latents
                    + transport_t.unsqueeze(-1) * transport_eps
                )
                flat_transport_noised_latents = transport_noised_latents.reshape(
                    -1,
                    transport_noised_latents.shape[-1],
                )
                flat_transport_t = transport_t.reshape(-1)

                transport_predicted_noise = critic(
                    flat_transport_noised_latents,
                    flat_transport_t,
                ).reshape_as(transport_noised_latents)
                transport_prior_score = prior_config.analytic_score(
                    flat_transport_noised_latents.float(),
                    flat_transport_t.float(),
                ).to(dtype=flat_transport_noised_latents.dtype).reshape_as(
                    transport_noised_latents
                )
                transport_pullback_weight = (
                    transport_config.kl_weight_schedule.pullback_weight(
                        flat_transport_t.float(),
                    )
                    .to(dtype=flat_transport_noised_latents.dtype)
                    .reshape_as(transport_t)
                )
                transport_field_terms = transport_pullback_weight.unsqueeze(-1) * (
                    transport_prior_score
                    + transport_predicted_noise / transport_t.unsqueeze(-1)
                )

                if transport_config.antipodal_estimate:
                    midpoint = transport_config.num_time_samples
                    transport_field_terms = 0.5 * (
                        transport_field_terms[:, :midpoint]
                        + transport_field_terms[:, midpoint:]
                    )

                transport_field = transport_field_terms.mean(dim=1)
                transport_field_norm = transport_field.norm(dim=-1, keepdim=True)
                transport_step_size = torch.minimum(
                    torch.full_like(
                        transport_field_norm,
                        transport_config.transport_step_size,
                    ),
                    torch.full_like(
                        transport_field_norm,
                        transport_config.transport_step_cap,
                    )
                    / transport_field_norm.clamp_min(1e-6),
                )
                transport_step = transport_step_size * transport_field
                transported_latents = chart_data_latents.detach() + transport_step

            encoder_transport_loss = F.mse_loss(
                chart_data_latents,
                transported_latents.detach(),
            )
            decoder_transport_loss = F.mse_loss(
                decoder(transported_latents),
                chart_data_batch.detach(),
            )
            generated_log_likelihood = runtime_data_config.log_likelihood(
                chart_prior_reconstruction.float(),
            ).mean()
            chart_loss = constraint_loss
            chart_loss = chart_loss + (
                transport_config.encoder_transport_weight * encoder_transport_loss
            )
            chart_loss = chart_loss + (
                transport_config.decoder_transport_weight * decoder_transport_loss
            )

        fabric.backward(chart_loss)
        fabric.clip_gradients(
            chart_transport_model,
            optimizer,
            max_norm=architecture_config.grad_clip_norm,
        )
        optimizer.step()

        if isinstance(constraint_method, LagrangianConstraintConfig):
            data_dual = (
                data_dual
                + constraint_method.dual_variable_lr
                * (data_cycle_loss.detach() - constraint_method.data_constraint_budget)
            ).clamp_min(0.0)
            prior_dual = (
                prior_dual
                + constraint_method.dual_variable_lr
                * (prior_cycle_loss.detach() - constraint_method.prior_constraint_budget)
            ).clamp_min(0.0)

        latest_chart_metrics = {
            "chart_loss": chart_loss.detach().item(),
            "encoder_transport_loss": encoder_transport_loss.detach().item(),
            "decoder_transport_loss": decoder_transport_loss.detach().item(),
            "transport_field_norm": transport_field_norm.mean().detach().item(),
            "avg_generated_log_likelihood": generated_log_likelihood.detach().item(),
            "data_cycle_loss": data_cycle_loss.detach().item(),
            "prior_cycle_loss": prior_cycle_loss.detach().item(),
        }
        metrics.update(latest_chart_metrics)
        metrics["data_dual"] = data_dual.detach().item()
        metrics["prior_dual"] = prior_dual.detach().item()

    integrated_history.append(metrics)
    integrated_progress.set_postfix(
        critic_loss=f"{metrics['critic_loss']:.4f}",
        chart_loss=f"{metrics['chart_loss']:.4f}",
    )

    if should_log_monitor(
        step=step,
        total_steps=training_config.integrated_n_steps,
        every_n_steps=monitor_config.log_every_n_steps_integrated,
    ):
        fabric.print(
            f"[integrated] step {step}/{training_config.integrated_n_steps}: {format_metrics_summary(metrics)}"
        )

        encoder_was_training = encoder.training
        decoder_was_training = decoder.training
        critic_was_training = critic.training
        encoder.eval()
        decoder.eval()
        critic.eval()

        with torch.no_grad():
            with runtime_precision_context():
                pair_samples, pair_labels = sample_monitor_pairs_batch(
                    total_batch_size=monitor_config.constraint_monitor_config.plot_n_sample_pairs,
                )
                pair_latents = encoder(pair_samples)
                pair_reconstructions = decoder(pair_latents)

                latent_samples, latent_labels = sample_monitor_batch(
                    batch_size_per_mode=monitor_config.constraint_monitor_config.plot_n_data_latents_per_mode,
                )
                latent_values = encoder(latent_samples)

                dense_samples, dense_labels = sample_monitor_batch(
                    batch_size_per_mode=monitor_config.critic_monitor_config.plot_n_data_latents_per_mode,
                )
                vector_samples, vector_labels = sample_monitor_batch(
                    batch_size_per_mode=monitor_config.critic_monitor_config.plot_n_vectors_per_mode,
                )
                dense_clean_latents = encoder(dense_samples).float()
                vector_clean_latents = encoder(vector_samples).float()

                score_snapshots = []
                for t_value in monitor_config.critic_monitor_config.sample_t_values:
                    cloud_latents, _ = sample_critic_snapshot(
                        clean_latents=dense_clean_latents,
                        t_value=t_value,
                    )
                    arrow_latents, data_score = sample_critic_snapshot(
                        clean_latents=vector_clean_latents,
                        t_value=t_value,
                    )
                    score_snapshots.append(
                        (
                            t_value,
                            cloud_latents.detach().cpu().float(),
                            arrow_latents.detach().cpu().float(),
                            data_score.detach().cpu().float(),
                        )
                    )

                spectrum_t_values = critic_spectrum_t_values()
                transport_field = estimate_clean_transport_field(
                    clean_latents=vector_clean_latents,
                    t_values=spectrum_t_values,
                )
                transport_grid_points, transport_grid_xs, transport_grid_ys = build_latent_grid(
                    points=dense_clean_latents,
                    resolution=31,
                )
                transport_grid_field = estimate_clean_transport_field(
                    clean_latents=transport_grid_points,
                    t_values=spectrum_t_values,
                )
                transport_grid_projection = project_latent_vectors_to_pca_plane(
                    reference_points=dense_clean_latents,
                    vectors=transport_grid_field,
                ).norm(dim=-1).reshape(
                    transport_grid_ys.shape[0],
                    transport_grid_xs.shape[0],
                )
                loss_spectrum = estimate_critic_loss_spectrum(
                    clean_latents=dense_clean_latents,
                    t_values=spectrum_t_values,
                )

                generated_prior_samples = prior_config.sample(
                    batch_size=monitor_config.integrated_monitor_config.plot_n_generated_samples,
                ).to(device=device, dtype=torch.float32)
                generated_samples = decoder(generated_prior_samples)
                generated_data_samples, generated_data_labels = sample_monitor_batch(
                    batch_size_per_mode=monitor_config.integrated_monitor_config.plot_n_data_samples_per_mode,
                )

            pair_samples_cpu = pair_samples.detach().cpu().float()
            pair_reconstructions_cpu = pair_reconstructions.detach().cpu().float()
            pair_labels_cpu = pair_labels.detach().cpu().long()
            pair_projected_samples, _, _ = data_config.decompose_projection(
                pair_samples_cpu,
            )
            pair_projected_reconstructions, _, _ = data_config.decompose_projection(
                pair_reconstructions_cpu,
            )
            pair_manifold_deviation = (
                (pair_reconstructions - pair_samples)
                .reshape(pair_samples.shape[0], -1)
                .norm(dim=-1)
                .detach()
                .cpu()
                .float()
            )

            latent_samples_cpu = latent_samples.detach().cpu().float()
            latent_values_cpu = latent_values.detach().cpu().float()
            latent_labels_cpu = latent_labels.detach().cpu().long()
            _, _, latent_off_plane = data_config.decompose_projection(
                latent_samples_cpu,
            )
            latent_off_manifold_norm = latent_off_plane.norm(dim=-1).float()

            dense_clean_latents_cpu = dense_clean_latents.detach().cpu().float()
            dense_labels_cpu = dense_labels.detach().cpu().long()
            vector_clean_latents_cpu = vector_clean_latents.detach().cpu().float()
            vector_labels_cpu = vector_labels.detach().cpu().long()
            transport_field_cpu = transport_field.detach().cpu().float()
            transport_grid_xs_cpu = transport_grid_xs.detach().cpu().float()
            transport_grid_ys_cpu = transport_grid_ys.detach().cpu().float()
            transport_grid_projection_cpu = (
                transport_grid_projection.detach().cpu().float()
            )
            generated_samples_projected, _, _ = data_config.decompose_projection(
                generated_samples.detach().cpu().float(),
            )
            generated_data_samples_projected, _, _ = data_config.decompose_projection(
                generated_data_samples.detach().cpu().float(),
            )
            generated_data_labels_cpu = generated_data_labels.detach().cpu().long()
            recent_history = pl.DataFrame(
                integrated_history[-monitor_config.integrated_monitor_config.keep_loss_last_n:]
            )

        if encoder_was_training:
            encoder.train()
        if decoder_was_training:
            decoder.train()
        if critic_was_training:
            critic.train()

        write_constraint_monitor_artifacts(
            step=step,
            stage="integrated",
            plot_prefix="constraint_",
            pair_projected_samples=pair_projected_samples,
            pair_projected_reconstructions=pair_projected_reconstructions,
            pair_manifold_deviation=pair_manifold_deviation,
            pair_labels=pair_labels_cpu,
            latent_values=latent_values_cpu,
            latent_off_manifold_norm=latent_off_manifold_norm,
            latent_labels=latent_labels_cpu,
        )
        write_critic_monitor_artifacts(
            step=step,
            stage="integrated",
            plot_prefix="critic_",
            dense_clean_latents=dense_clean_latents_cpu,
            dense_labels=dense_labels_cpu,
            vector_clean_latents=vector_clean_latents_cpu,
            vector_labels=vector_labels_cpu,
            score_snapshots=score_snapshots,
            transport_field=transport_field_cpu,
            transport_grid_xs=transport_grid_xs_cpu,
            transport_grid_ys=transport_grid_ys_cpu,
            transport_grid_projection=transport_grid_projection_cpu,
            loss_spectrum=loss_spectrum,
            num_contour_lines=monitor_config.critic_monitor_config.num_contour_lines,
            score_snapshot_subdir="score_snapshots",
        )
        write_integrated_monitor_artifacts(
            step=step,
            stage="integrated",
            generated_samples=generated_samples_projected,
            generated_data_samples=generated_data_samples_projected,
            generated_data_labels=generated_data_labels_cpu,
            recent_history=recent_history,
        )

integrated_history = pl.DataFrame(integrated_history)
display(integrated_history.tail())

integrated:   0%|          | 0/1000000 [00:00<?, ?it/s]

[integrated] step 1/1000000: critic_loss=0.0058, chart_loss=0.0052, encoder_transport_loss=0.0000, decoder_transport_loss=0.0020, transport_field_norm=2.9795, avg_generated_log_likelihood=-34.6942, data_cycle_loss=0.0011, prior_cycle_loss=0.0021, data_dual=0.0000, prior_dual=0.0000
[integrated] step 4000/1000000: critic_loss=0.0219, chart_loss=0.0010, encoder_transport_loss=0.0000, decoder_transport_loss=0.0003, transport_field_norm=3.7680, avg_generated_log_likelihood=-15.3146, data_cycle_loss=0.0001, prior_cycle_loss=0.0006, data_dual=0.0000, prior_dual=0.0000
[integrated] step 8000/1000000: critic_loss=0.0359, chart_loss=0.0013, encoder_transport_loss=0.0000, decoder_transport_loss=0.0005, transport_field_norm=5.3298, avg_generated_log_likelihood=-12.8010, data_cycle_loss=0.0002, prior_cycle_loss=0.0006, data_dual=0.0000, prior_dual=0.0000
[integrated] step 12000/1000000: critic_loss=0.0561, chart_loss=0.0020, encoder_transport_loss=0.0000, decoder_transport_loss=0.0010, transport_f